# Shared setup for the hardware experiments

The experiment notebooks run this setup in their first cell. It creates the same fixed SNP problem, QAOA circuit, backend and scoring functions for each comparison. Read the cells here if you want to see exactly what the experiments share.

The five-qubit data and angles below are the earlier pilot snapshot. The angles were locally reproduced and still need team confirmation. The experiments do not retrain them.

## Inputs

Change the reads, fixed angles and backend together for another problem. Missing read entries are `-1`. A backend must have enough qubits; FakeManilaV2 has five.

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

hardware_inputs = globals().get('hardware_inputs', {})
reads = np.asarray(hardware_inputs.get('reads', [
    [0, 1, 1, -1, -1], [1, 0, 0, -1, -1], [-1, 1, 1, 0, -1],
    [-1, 0, 0, 1, -1], [-1, -1, 1, 0, 1], [-1, -1, 0, 1, 0],
    [0, -1, -1, 0, 1], [1, -1, -1, 1, 0], [0, 0, -1, -1, -1],
]))
gammas = hardware_inputs.get('gammas', [2.7773446205747447])
betas = hardware_inputs.get('betas', [2.0658685280892244])
backend = hardware_inputs.get('backend') or FakeManilaV2()
shots = 2048
compile_seeds = list(range(20))
noise_seeds = compile_seeds[:3]
noise_repeats = 3

## Build the evidence and cost

For each observed pair, `agree` counts equal values and `disagree` counts different values. The cost counts votes that disagree with a candidate bitstring.

In [ ]:
def configure_problem(new_reads, new_gammas, new_betas, backend_override=None, evidence=None):
    """Reuse frozen standard-QAOA angles on a shared input; never train or submit jobs."""
    global reads, gammas, betas, backend, n, agree, disagree, couplings, edges, scale, optimum
    data = np.asarray(new_reads)
    selected_backend = backend if backend_override is None else backend_override
    if data.ndim != 2 or not np.isin(data, [-1, 0, 1]).all():
        raise ValueError('Reads must contain only -1, 0 and 1.')
    size = data.shape[1]
    if size < 1 or size > selected_backend.num_qubits:
        raise ValueError('Choose a backend with enough qubits for these reads.')
    gamma, beta = list(new_gammas), list(new_betas)
    if not gamma or len(gamma) != len(beta) or not np.isfinite(gamma + beta).all():
        raise ValueError('Provide one finite gamma and beta per layer.')
    a = np.zeros((size, size), dtype=float)
    d = np.zeros_like(a)
    if evidence is None:
        for read in data:
            for i, j in itertools.combinations(np.flatnonzero(read >= 0), 2):
                (a if read[i] == read[j] else d)[i, j] += 1
    else:
        a, d = (np.asarray(matrix, dtype=float) for matrix in evidence)
        if (a.shape != (size, size) or d.shape != a.shape
                or not np.isfinite(a).all() or not np.isfinite(d).all()
                or (a < 0).any() or (d < 0).any()
                or not np.array_equal(a, np.triu(a, 1))
                or not np.array_equal(d, np.triu(d, 1))):
            raise ValueError('Evidence must be finite nonnegative upper-triangular A, D.')
    reads, gammas, betas, backend, n = data.copy(), gamma, beta, selected_backend, size
    agree, disagree = a.copy(), d.copy()
    couplings = (disagree - agree) / 2
    edges = [(i, j) for i, j in itertools.combinations(range(n), 2) if couplings[i, j] != 0]
    scale = max(1., float(np.abs(couplings).max()))
    optimum = min(cost(bits) for bits in itertools.product([0, 1], repeat=n)) if n <= 12 else None


In [ ]:
def cost(bits):
    return sum(agree[i, j] if bits[i] != bits[j] else disagree[i, j]
               for i, j in itertools.combinations(range(n), 2))

configure_problem(reads, gammas, betas, evidence=hardware_inputs.get('evidence'))
print(f'{n} SNPs, {len(edges)} active interactions, backend: {backend.name}')
print('Exact minimum:', optimum if optimum is not None else 'not computed')

## Make the fixed circuit

`order` says which original SNP goes on each logical wire. The same interactions and angles are used in every ordering. Each measurement is written back to its original SNP bit, so the output does not need a second layout reversal.

In [4]:
def make_circuit(order=None, gate_order='original'):
    order = tuple(range(n)) if order is None else tuple(order)
    if len(order) != n or set(order) != set(range(n)):
        raise ValueError('Order must contain every SNP exactly once.')
    if gate_order not in ('original', 'reverse', 'magnitude'):
        raise ValueError('Unknown gate ordering.')
    wire = {snp: position for position, snp in enumerate(order)}
    ordered_edges = list(edges)
    if gate_order == 'reverse':
        ordered_edges.reverse()
    elif gate_order == 'magnitude':
        ordered_edges.sort(key=lambda edge: -abs(couplings[edge]))

    circuit = QuantumCircuit(n, n)
    circuit.h(range(n))
    for gamma, beta in zip(gammas, betas):
        for i, j in ordered_edges:
            circuit.rzz(2 * gamma * couplings[i, j] / scale, wire[i], wire[j])
        circuit.rx(2 * beta, range(n))
    for snp in range(n):
        circuit.measure(wire[snp], snp)
    return circuit

## Compare resources and samples

Depth and two-qubit counts are measured after compilation. A lower count may reduce exposure to gate errors, but it does not guarantee a better noisy result. `score` reads Qiskit keys once, in the original SNP order.

In [ ]:
def resources(circuit):
    return {
        '2q gates': sum(item.operation.num_qubits == 2
                        and item.operation.name != 'barrier' for item in circuit.data),
        '2q depth': circuit.depth(lambda item: item.operation.num_qubits == 2
                                 and item.operation.name != 'barrier'),
        'depth': circuit.depth(),
    }


def score(counts):
    total = sum(counts.values())
    values = [(cost([int(bit) for bit in key.replace(' ', '')[::-1]]), count)
              for key, count in counts.items()]
    return {
        'mean cost': sum(value * count for value, count in values) / total,
        'optimum frequency': (sum(count for value, count in values if np.isclose(value, optimum, rtol=0, atol=1e-9)) / total
                              if optimum is not None else np.nan),
    }

def boxplot_legend(fig):
    from matplotlib.lines import Line2D
    fig.legend(handles=[
        Line2D([0], [0], color='C1', label='Median'),
        Line2D([0], [0], color='green', marker='^', linestyle='None', label='Mean'),
    ], loc='upper center', bbox_to_anchor=(0.5, 0.94), ncol=2, frameon=False)


## Check measurement order

The tests below use a small exact statevector. Skip this reference for larger inputs: its memory use grows as 2^n.

In [6]:
def logical_probabilities(measured):
    mapping = [(measured.find_bit(item.qubits[0]).index,
                measured.find_bit(item.clbits[0]).index)
               for item in measured.data if item.operation.name == 'measure']
    probabilities = Statevector.from_instruction(
        measured.remove_final_measurements(inplace=False)).probabilities()
    result = np.zeros(2 ** n)
    for physical_state, probability in enumerate(probabilities):
        logical_state = sum(((physical_state >> q) & 1) << c for q, c in mapping)
        result[logical_state] += probability
    return result

if n <= 12:
    ideal = logical_probabilities(make_circuit())
    assert np.allclose(ideal, logical_probabilities(make_circuit(tuple(reversed(range(n))))))
    print('The reversed SNP order keeps the same ideal output distribution.')
else:
    ideal = None
    print('Exact statevector check skipped for this input size.')

The reversed SNP order keeps the same ideal output distribution.


## Shared-input adapter

Defaults retain Alex's original five-SNP pilot. Call `configure_problem(reads, gammas, betas, evidence=(A, D))` after loading this notebook, or supply `hardware_inputs` before `%run -i hardware_setup.ipynb`. Angles must be locally trained and frozen. `evidence` optionally selects another pairwise graph, including weighted or pruned evidence. `make_circuit` then preserves SNP-to-classical-bit mapping under permutations. For warm-start/multi-angle, compile the frozen circuit directly instead of forcing it into the standard gamma/beta interface. FakeManilaV2/Aer results are simulations, never QPU measurements.